# Local notebook -> Modal function calling

This notebook stays local in Cursor. Only the decorated function runs on a Modal GPU.

Before running the final cell: `uv sync --extra remote` and `uv run modal setup`. The remote and local Python versions are both 3.12.

In [ ]:
import modal

app = modal.App("ml-lab-notebook")
image = modal.Image.debian_slim(python_version="3.12").pip_install("torch")
print("Modal client loaded locally")

In [ ]:
@app.function(image=image, gpu="T4", timeout=300)
def remote_matmul(size: int = 1024) -> dict[str, object]:
    import torch

    left = torch.randn(size, size, device="cuda")
    right = torch.randn(size, size, device="cuda")
    result = left @ right
    return {
        "device": str(result.device),
        "gpu": torch.cuda.get_device_name(0),
        "shape": list(result.shape),
    }


print("Remote function defined locally")

In [ ]:
with modal.enable_output(), app.run():
    result = remote_matmul.remote(1024)

print(result)